# 06 FS2 LEAR

This notebook focuses on LEAR with `FS2`: lagged prices plus forecast-known calendar features. It reuses the shared LEAR benchmark run and interprets the `FS2` view inside the same standardized reporting structure.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Image, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig, MonitoringConfig
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.notebook_support import (
    apply_standard_matplotlib_style,
    build_compared_models_overview,
    build_dm_summary_table,
    build_model_style_map,
    build_week_metrics_for_predictions,
    build_reporting_metric_grid,
    build_reporting_summary_table,
    build_runtime_summary_table,
    estimate_run_duration_seconds,
    format_duration,
    load_selected_case_weeks,
    load_standard_report_bundle,
    render_plot_gallery,
    render_reporting_metric_dashboard,
    run_suite_with_feedback,
    style_dm_summary_table,
    style_model_overview_table,
    style_reporting_metric_grid,
    style_reporting_summary_table,
    style_runtime_summary_table,
    summarize_timing_compact,
    write_actual_vs_predicted_scatter_plot,
    write_horizon_error_plot,
    write_mae_by_hour_of_day_plot,
    write_residual_distribution_plot,
    write_standard_week_selection_plots,
    write_week_plots_for_models,
)

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root
run_root = output_root / "runs"


def latest_run_matching(pattern: str) -> Path:
    matches = sorted(run_root.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No run directories found for pattern: {pattern}")
    return matches[-1]


This notebook reuses the shared LEAR rerun cell from the `FS1` stage, then narrows the interpretation to `FS2`, the clean calendar-augmented reference feature set.


In [ ]:
ALLOW_HEAVY_RERUN = False

if ALLOW_HEAVY_RERUN:
    from dataclasses import replace

    from dataclasses import replace
    from hourly_da.models.lear import LEARModel, LEARSettings
    from hourly_da.models.naive import PreviousWeekNaiveModel
    rerun_config = replace(config)

    rerun_config = replace(config, monitoring=MonitoringConfig(fit_time_absolute_threshold_sec=5.0))
    models = [
        PreviousWeekNaiveModel(),
        LEARModel(LEARSettings(fs_level="FS1", training_window_hours=90 * 24, min_train_rows=30 * 24, alpha=0.01)),
        LEARModel(LEARSettings(fs_level="FS2", training_window_hours=90 * 24, min_train_rows=30 * 24, alpha=0.01)),
    ]
    rerun_payload = run_suite_with_feedback(
        config=rerun_config,
        run_label="lear_benchmark",
        models=models,
        include_external_features=False,
        show_progress=True,
        progress_label="lear_benchmark",
    )
    print(rerun_payload["official_naive"])
else:
    print("Rerun skipped. Set ALLOW_HEAVY_RERUN = True to execute the shared benchmark pipeline from this notebook.")


In [ ]:
run_dir = find_latest_run(output_root, "lear_benchmark")
print(run_dir)


The cells below use the shared **standardized thesis reporting layer**. The forecasting logic stays unchanged; only the post-run interpretation is organized into one consistent reporting sequence.


In [ ]:
apply_standard_matplotlib_style()

current_suite_models = load_json(run_dir, "suite_models.json")["models"]
current_model_names = [record.get("name", record.get("model")) for record in current_suite_models]

COMPARISON_MODEL_SPECS = [{'model': 'naive_previous_week', 'display_name': 'Naive previous week', 'description': 'Official naive benchmark carried into the active benchmark runs.', 'role': 'benchmark'}, {'model': 'sarima', 'run_label': 'sarima_benchmark', 'display_name': 'SARIMA', 'description': 'Key prior FS1 time-series benchmark.', 'role': 'prior'}, {'model': 'lear_fs1', 'display_name': 'LEAR FS1', 'description': 'Same-model FS1 benchmark from the current LEAR run.', 'role': 'benchmark'}, {'model': 'lear_fs2', 'display_name': 'LEAR FS2', 'description': 'LEAR with lagged prices plus calendar features.', 'role': 'current'}]
HORIZON_PLOT_MODELS = ['naive_previous_week', 'sarima', 'lear_fs1', 'lear_fs2']
WEEK_PLOT_MODELS = ['naive_previous_week', 'sarima', 'lear_fs1', 'lear_fs2']
DIAGNOSTIC_MODELS = ['naive_previous_week', 'lear_fs1', 'lear_fs2']
DM_CHALLENGER_MODELS = ['lear_fs2']

report_bundle = load_standard_report_bundle(
    output_root=output_root,
    current_run_dir=run_dir,
    comparison_specs=COMPARISON_MODEL_SPECS,
)
model_styles = build_model_style_map(
    report_bundle["comparison_specs"],
    official_naive_model=str(report_bundle["official_naive"]["model"]),
)
report_output_dir = output_root / "notebook_artifacts" / "06_fs2_lear" / run_dir.name / "standard_report"
report_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Current run: {run_dir.name}")
print(f"Current models: {current_model_names}")
print(f"Official naive benchmark: {report_bundle['official_naive']['model']}")


## 1. Short overview of compared models


In [ ]:
overview_table = build_compared_models_overview(
    report_bundle["comparison_specs"],
    official_naive_model=str(report_bundle["official_naive"]["model"]),
)
display(style_model_overview_table(overview_table))
display(pd.DataFrame([report_bundle["official_naive"]]))


## 2. Main validation and test summary tables

Each table reports the same thesis metrics for the three standard reporting slices:
- `D only`
- `Guidance only`
- `Stitched all-horizon`


In [ ]:
validation_summary = build_reporting_summary_table(
    report_bundle["metrics_by_reporting_level"],
    split_name="validation",
    model_order=report_bundle["model_order"],
)
test_summary = build_reporting_summary_table(
    report_bundle["metrics_by_reporting_level"],
    split_name="test",
    model_order=report_bundle["model_order"],
)

if not validation_summary.empty:
    display(style_reporting_summary_table(validation_summary, caption="Validation summary"))
if not test_summary.empty:
    display(style_reporting_summary_table(test_summary, caption="Test summary"))


## 3. By-horizon error view

This figure keeps the split fixed and shows how MAE changes from `D` through `D+4`.


In [ ]:
horizon_plot_path = write_horizon_error_plot(
    metrics_by_lead_day=report_bundle["metrics_by_lead_day"][
        report_bundle["metrics_by_lead_day"]["model"].isin(HORIZON_PLOT_MODELS)
    ].copy(),
    output_path=report_output_dir / "horizon_error_mae.png",
    model_order=HORIZON_PLOT_MODELS,
    model_styles=model_styles,
)
if horizon_plot_path is not None:
    display(Image(filename=str(horizon_plot_path)))


## 4. Forecast vs actual on the frozen week selections

These plots reuse the objectively selected weeks from notebook `00`. For a clean visual comparison, the overlays use the operational `D only` path so each target hour appears once.


In [ ]:
selection_run_dir, selected_weeks = load_selected_case_weeks(output_root)
display(selected_weeks[["category", "iso_week_id", "week_start_local_date", "week_end_local_date"]])

model_label_map = (
    report_bundle["comparison_specs"][["model", "display_name"]]
    .drop_duplicates(subset=["model"])
    .set_index("model")["display_name"]
    .to_dict()
)
week_metrics = build_week_metrics_for_predictions(report_bundle["predictions_long"], config, selected_weeks)
week_metrics_display = (
    week_metrics[week_metrics["model"].isin(WEEK_PLOT_MODELS)]
    .assign(Model=lambda frame: frame["model"].map(model_label_map).fillna(frame["model"]))
    [["category", "iso_week_id", "Model", "mae", "rmse", "bias", "coverage_pct", "max_abs_error"]]
    .sort_values(["category", "mae", "Model"])
    .reset_index(drop=True)
)
display(
    week_metrics_display.style
    .format(
        {
            "mae": "{:.2f}",
            "rmse": "{:.2f}",
            "bias": "{:+.2f}",
            "coverage_pct": "{:.2f}%",
            "max_abs_error": "{:.2f}",
        }
    )
    .hide(axis="index")
)

week_plot_paths = write_standard_week_selection_plots(
    predictions=report_bundle["predictions_long"],
    config=config,
    selected_weeks=selected_weeks,
    output_dir=report_output_dir / "week_plots",
    model_order=WEEK_PLOT_MODELS,
    model_styles=model_styles,
    split_name="test",
    reporting_level="d_only",
    title_prefix='FS2 LEAR, D-only path',
)
display(render_plot_gallery(week_plot_paths, columns=2))


## 5. Diagnostic plots

The diagnostics below focus on the **test split** and the same `D only` operational path as the week overlays, so the interpretation is based on unique forecast-target pairs rather than repeated multi-origin horizons.


In [ ]:
diagnostic_plot_paths = []

mae_by_hour_path = write_mae_by_hour_of_day_plot(
    predictions=report_bundle["predictions_long"],
    config=config,
    output_path=report_output_dir / "diagnostics" / "mae_by_hour_of_day.png",
    model_order=DIAGNOSTIC_MODELS,
    model_styles=model_styles,
)
if mae_by_hour_path is not None:
    diagnostic_plot_paths.append(mae_by_hour_path)

residual_path = write_residual_distribution_plot(
    predictions=report_bundle["predictions_long"],
    config=config,
    output_path=report_output_dir / "diagnostics" / "residual_distribution.png",
    model_order=DIAGNOSTIC_MODELS,
    model_styles=model_styles,
)
if residual_path is not None:
    diagnostic_plot_paths.append(residual_path)

scatter_path = write_actual_vs_predicted_scatter_plot(
    predictions=report_bundle["predictions_long"],
    config=config,
    output_path=report_output_dir / "diagnostics" / "actual_vs_predicted.png",
    model_order=DIAGNOSTIC_MODELS,
    model_styles=model_styles,
)
if scatter_path is not None:
    diagnostic_plot_paths.append(scatter_path)

display(render_plot_gallery(diagnostic_plot_paths, columns=2))


## 6. Statistical comparison

The Diebold-Mariano table is interpreted as follows:
- negative DM statistic favors the challenger
- positive DM statistic favors the benchmark
- the verdict column applies a `p < 0.05` threshold


In [ ]:
dm_display_order = [
    row["display_name"]
    for row in report_bundle["comparison_specs"].to_dict(orient="records")
    if row["model"] in DM_CHALLENGER_MODELS
]
dm_summary = build_dm_summary_table(
    report_bundle["diebold_mariano_by_reporting_level"][
        report_bundle["diebold_mariano_by_reporting_level"]["challenger_model"].isin(DM_CHALLENGER_MODELS)
    ].copy(),
    challenger_display_order=dm_display_order,
)
if dm_summary.empty:
    print("No configured Diebold-Mariano comparisons were available for this notebook.")
else:
    display(style_dm_summary_table(dm_summary))


## 7. Runtime and practicality summary


In [ ]:
runtime_summary = build_runtime_summary_table(
    report_bundle["timing_summary"],
    model_order=report_bundle["model_order"],
)
if runtime_summary.empty:
    print("No runtime summary was available for this notebook.")
else:
    display(style_runtime_summary_table(runtime_summary))
